# conv-channel-sum — ex1: verify conv2d contracts the IC axis

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `conv-channel-sum`. Running the final beacon cell reports progress against the `CNN: Channel-axis sum semantics` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Channel-axis sum semantics` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-channel-sum`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-channel-sum"
DD_SUBTOPIC = "CNN: Channel-axis sum semantics"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Conv2d channel-axis sum semantics — quick refresher

A 2-D convolution does **three** things at once. The cleanest way to see this is the einsum form:

```
y = einops.einsum(
    x_windows, weight,
    'b ic oh ow kh kw, oc ic kh kw -> b oc oh ow',
)
```

**Read the einsum letter-by-letter:**
- `b` (batch) — passes through.
- `oc` (output channels) — appears only on the right of the kernel; each output channel is a separate filter.
- `ic` (input channels) — appears on **both** inputs but NOT on the output. That's a sum: every output pixel is the sum across `IC` of `window_ic * kernel_ic`. The kernel-output gets one scalar per (oh, ow).
- `kh`, `kw` — also contracted (sum) between window and kernel.
- `oh`, `ow` — pass through from the windowed input.

**The headline.** Convolution is a *per-output-channel* operation that sums across input channels. With `IC = 3` and `OC = 16`, you have 16 independent filters, each of which mixes the 3 RGB channels into a single scalar at every spatial location. The contraction axis is `IC` (plus the kernel spatial extent).

### Exercise 1 — verify conv2d contracts the IC axis

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the per-output-channel sum semantics of conv2d by decomposing `F.conv2d` into a sum of `IC` single-channel convolutions and verifying numerical equivalence.
> Keywords: conv2d, einsum, channel-sum, contraction
> ```

**KCs targeted:** `conv-ic-contraction`, `conv-per-oc-filter`

Implement `ex1_conv2d_by_ic_sum(x, weight)`. Given input `x: (B, IC, H, W)` and kernel `weight: (OC, IC, KH, KW)`, compute the same result as `F.conv2d(x, weight)` (stride 1, no padding) but **only by looping over `IC` and summing single-channel convolutions**.

**Algorithm:**
1. Allocate an output tensor `y` of the right shape (`(B, OC, H-KH+1, W-KW+1)`).
2. For each `ic` in `range(IC)`:
   - Slice `x[:, ic:ic+1, :, :]` — one input channel kept as a size-1 axis: `(B, 1, H, W)`.
   - Slice `weight[:, ic:ic+1, :, :]` — one input-channel kernel tap kept as a size-1 axis: `(OC, 1, KH, KW)`.
   - Run `F.conv2d` on this single-channel pair and **add** the result into `y`.
3. Return `y`.

**The point of the drill.** Decomposing the IC contraction manually makes it obvious that conv2d is *per-OC, summed across IC* — the IC axis is contracted, not preserved.

The test compares your output to `F.conv2d(x, weight)` to fp tolerance.

In [ ]:
def ex1_conv2d_by_ic_sum(x: Tensor, weight: Tensor) -> Tensor:
    """conv2d, but reconstructed as a sum over input channels."""
    raise NotImplementedError()


def _test_ex1():
    from torch.nn import functional as F

    rng = t.Generator().manual_seed(0)

    # Small smoke test: B=1, IC=3 (RGB-like), OC=2, K=3.
    B, IC, H, W = 1, 3, 8, 8
    OC, KH, KW = 2, 3, 3
    x = t.randn(B, IC, H, W, generator=rng)
    weight = t.randn(OC, IC, KH, KW, generator=rng)
    y_ours = ex1_conv2d_by_ic_sum(x, weight)
    y_ref  = F.conv2d(x, weight)
    assert y_ours.shape == y_ref.shape == (B, OC, H - KH + 1, W - KW + 1), (
        f'shape mismatch: ours={tuple(y_ours.shape)} ref={tuple(y_ref.shape)}'
    )
    assert y_ours.dtype == t.float32
    assert t.allclose(y_ours, y_ref, atol=1e-4), 'IC-sum decomposition must equal F.conv2d'

    # Larger spec — confirms contraction holds for many IC.
    B, IC, H, W = 2, 7, 12, 12
    OC, KH, KW = 4, 5, 5
    x2 = t.randn(B, IC, H, W, generator=rng)
    w2 = t.randn(OC, IC, KH, KW, generator=rng)
    y2 = ex1_conv2d_by_ic_sum(x2, w2)
    y2_ref = F.conv2d(x2, w2)
    assert t.allclose(y2, y2_ref, atol=1e-4)

    # Sanity probe: zero out one IC slot of the kernel and confirm the
    # output drops by exactly that channel's contribution.
    w_masked = w2.clone()
    w_masked[:, 0, :, :] = 0.0  # kill ic=0
    y_masked = ex1_conv2d_by_ic_sum(x2, w_masked)
    y_only0  = F.conv2d(x2[:, 0:1], w2[:, 0:1])
    assert t.allclose(y_masked + y_only0, y2_ref, atol=1e-4), (
        'killing ic=0 in the kernel must subtract exactly that channel\'s contribution'
    )

    # IC=1 edge case — should be a no-op channel sum (one iteration only).
    x1 = t.randn(1, 1, 6, 6, generator=rng)
    w1 = t.randn(3, 1, 2, 2, generator=rng)
    out1 = ex1_conv2d_by_ic_sum(x1, w1)
    assert t.allclose(out1, F.conv2d(x1, w1), atol=1e-5)
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_conv2d_by_ic_sum(x: Tensor, weight: Tensor) -> Tensor:
    from torch.nn import functional as F
    B, IC, H, W = x.shape
    OC, _, KH, KW = weight.shape
    y = t.zeros(B, OC, H - KH + 1, W - KW + 1, dtype=x.dtype)
    for ic in range(IC):
        y = y + F.conv2d(x[:, ic:ic+1], weight[:, ic:ic+1])
    return y
```

**Why slicing with `ic:ic+1` (not `[ic]`).** Keeping a size-1 axis preserves the 4-D layout `(B, 1, H, W)` that `F.conv2d` expects. Using `[ic]` would collapse to `(B, H, W)` and crash.

**Why `y = y + ...` (not `y +=`).** `+=` would be an in-place op on a freshly allocated zero tensor — fine here, but writing the non-in-place form makes the accumulation obvious and autograd-safe if you ever want to backprop through this.

**The bigger picture.** This is exactly what the einsum form compiles to: `'b ic h w, oc ic kh kw -> b oc h_out w_out'` says "sum over `ic` (and `kh, kw`)." The loop makes the contraction explicit; the einsum makes it fast.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()